# GPU Preprocessing Runner (Colab)

This notebook runs the project CLI scripts from a Google Colab GPU runtime.

What you need to edit:
- `REPO_DIR` (where the repository is located in Colab)
- `INPUT_DIR` (folder with your 2D slices)

Pipeline executed by the main script:
1. Stack slices into a 3D volume
2. Apply norm200 normalization
3. Run CUDA NLM (chunked)
4. Save outputs into `norm200_output/` and `nlm_output/`

## 1) Runtime setup
Use a GPU runtime in Colab: Runtime -> Change runtime type -> GPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version)
print('Working dir:', os.getcwd())

In [ ]:
# Optional but recommended: verify GPU is attached
!nvidia-smi

In [ ]:
# Install runtime dependencies
# In Colab, torch usually exists already, but this ensures required packages are present.
!pip -q install numpy tifffile torch

## 2) Edit paths only
Set your repository path and input slices path below.

In [ ]:
# TODO: edit these two paths
REPO_DIR = Path('/content/nnUNet4SoilXrayCT')
INPUT_DIR = Path('/content/path_to_slices')

print('REPO_DIR =', REPO_DIR)
print('INPUT_DIR =', INPUT_DIR)

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Repository not found: {REPO_DIR}')
if not INPUT_DIR.exists():
    raise FileNotFoundError(f'Input folder not found: {INPUT_DIR}')

## 3) Quick CLI check
This verifies the script is callable and shows CLI help.

In [ ]:
cmd = ['python', 'preprocess/run_preprocess.py', '--help']
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)

## 4) Run the main GPU preprocessing CLI
This executes stack -> norm200 -> CUDA NLM (chunked).

In [ ]:
cmd = [
    'python',
    'preprocess/run_preprocess.py',
    '--input_dir',
    str(INPUT_DIR),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=str(REPO_DIR), check=True)

## 5) Validate output files
Expected outputs:
- `norm200_output/norm200_volume.tif`
- `nlm_output/nlm_volume.tif`

In [ ]:
import tifffile

norm_path = REPO_DIR / 'norm200_output' / 'norm200_volume.tif'
nlm_path = REPO_DIR / 'nlm_output' / 'nlm_volume.tif'

print('norm path exists:', norm_path.exists(), norm_path)
print('nlm path exists:', nlm_path.exists(), nlm_path)

if norm_path.exists():
    norm_vol = tifffile.imread(norm_path)
    print('norm200 shape:', norm_vol.shape, 'dtype:', norm_vol.dtype)

if nlm_path.exists():
    nlm_vol = tifffile.imread(nlm_path)
    print('nlm shape:', nlm_vol.shape, 'dtype:', nlm_vol.dtype)

## Optional: run the playground CLI (local desktop workflow)
This command is provided for completeness. It is not recommended in Colab because Napari is a desktop viewer.

In [ ]:
# Example only (typically not used in Colab):
# subprocess.run([
#     'python', 'preprocess_playground/run_napari_filters.py',
#     '--input_dir', str(INPUT_DIR),
#     '--filter', 'nlm'
# ], cwd=str(REPO_DIR), check=True)